In [1]:
import bpy
import bmesh
import numpy as np
import matplotlib.pyplot as plt
import import_ipynb
import itertools

from L001_hello_world import render, clear_scene
from L004_pixel_measurement_model import add_sensor_noise, get_sensor_noise_statistics

# Clear scene
if __name__ == '__main__':
    clear_scene()

In [2]:
if __name__ == '__main__':
    # Set render engine and resolution for ray tracing
    bpy.context.scene.render.engine = 'CYCLES'
    bpy.context.scene.render.resolution_x = 512
    bpy.context.scene.render.resolution_y = 512
    bpy.context.scene.render.image_settings.media_type = 'VIDEO'
    bpy.context.scene.render.ffmpeg.format = 'MPEG4'
    bpy.context.scene.render.use_file_extension = False

    # Do not reduce number of samples per pixel based on render noise
    bpy.context.scene.cycles.use_adaptive_sampling = False 

    # Set number of samples to render at each pixel
    bpy.context.scene.cycles.samples = 10

    # Do not denoise the rendered image
    bpy.context.scene.cycles.use_denoising = False

    # Set film filter to box
    bpy.context.scene.cycles.pixel_filter_type = 'BOX'

In [3]:
if __name__ == '__main__': 
    world = bpy.data.worlds.new("World")
    scene = bpy.context.scene
    scene.world = world
    world.node_tree.nodes["Background"].inputs[1].default_value = 10


    # Create camera
    bpy.ops.object.camera_add(location=(0, 0, 5),
                              rotation=(0, 0, 0))
    cam = bpy.context.active_object
    bpy.context.scene.camera = cam


    # Add plane to scene
    bpy.ops.mesh.primitive_plane_add(location=(0, 0, 0), 
                                     rotation=(0, 0, 0))
    wheel = bpy.context.active_object


    # Create material
    mat = bpy.data.materials.new(name="TexturedPlaneMaterial")
    nodes = mat.node_tree.nodes
    links = mat.node_tree.links
    nodes.clear()

    # Nodes
    output = nodes.new(type="ShaderNodeOutputMaterial")
    bsdf = nodes.new(type="ShaderNodeBsdfTransparent")
    texcoord = nodes.new(type="ShaderNodeTexCoord")
    image_tex = nodes.new(type="ShaderNodeTexImage")

    # Load image
    N = 512
    image = bpy.data.images.new('pattern', N, N)
    image_data = np.zeros((N,N,4))

    X, Y = np.meshgrid(np.linspace(-1,1,N), np.linspace(-1,1,N))
    theta = np.mod(np.atan2(Y,X), 2*np.pi) / (2*np.pi)
    r = np.sqrt(X**2 + Y**2)

    image_data[:,:,0] = np.where((theta>=0.00) & (theta<0.25), 1, 0)
    image_data[:,:,1] = np.where((theta>=0.25) & (theta<0.50), 1, 0)
    image_data[:,:,2] = np.where((theta>=0.50) & (theta<0.75), 1, 0)

    image_data[:,:,0] = np.where(r>1, 1, image_data[:,:,0])
    image_data[:,:,1] = np.where(r>1, 1, image_data[:,:,1])
    image_data[:,:,2] = np.where(r>1, 1, image_data[:,:,2])

    image_data[:,:,3] = 1

    image.pixels = image_data.ravel()


    image_tex.image = image

    # Layout (optional, for readability)
    texcoord.location = (-900, 0)
    image_tex.location = (-600, 0)
    bsdf.location = (-300, 0)
    output.location = (0, 0)

    # Links
    links.new(texcoord.outputs["UV"], image_tex.inputs["Vector"])
    links.new(image_tex.outputs["Color"], bsdf.inputs["Color"])
    links.new(bsdf.outputs["BSDF"], output.inputs["Surface"])

    # Assign material
    wheel.data.materials.append(mat)

    fcurve = wheel.driver_add("rotation_euler", 2)
    driver = fcurve.driver
    driver.expression = "frame*sin(frame*3.14159*2/60)"

In [4]:
from wurlitzer import pipes
from IPython.display import Video

path = './tmp/output.mp4'
bpy.context.scene.render.filepath = path

bpy.context.scene.frame_start = 1
bpy.context.scene.frame_end = 60

# Render the scene
with pipes():
    bpy.ops.render.render(animation=True)

Video("./tmp/output.mp4", embed=True, html_attributes="loop autoplay muted")

In [5]:
bpy.context.scene.render.use_motion_blur = True
bpy.context.scene.render.motion_blur_shutter = 1

path = './tmp/output.mp4'
bpy.context.scene.render.filepath = path

bpy.context.scene.frame_start = 1
bpy.context.scene.frame_end = 60

# Render the scene
with pipes():
    bpy.ops.render.render(animation=True)

Video("./tmp/output.mp4", embed=True, html_attributes="loop autoplay muted")

In [6]:
bpy.context.scene.cycles.rolling_shutter_type = 'TOP'
bpy.context.scene.cycles.rolling_shutter_duration = 0.2

path = './tmp/output.mp4'
bpy.context.scene.render.filepath = path

bpy.context.scene.frame_start = 1
bpy.context.scene.frame_end = 60

# Render the scene
with pipes():
    bpy.ops.render.render(animation=True)

Video("./tmp/output.mp4", embed=True, html_attributes="loop autoplay muted")